# Paderborn Bearing Fault Diagnosis — Main Pipeline

A complete end-to-end workflow for bearing fault classification using the Paderborn
University bearing dataset. This notebook ties together the four supporting modules
(`download_dataset`, `data_loader`, `dsp_features`, `ml_classification`)
into a single, reproducible pipeline.

**Dataset:** Paderborn University Bearing Data Center  
**Signals:** Phase currents (64 kHz), vibration (64 kHz), operating conditions (4 kHz)  
**Task:** 3-class fault classification — Healthy / Outer Race Fault / Inner Race Fault

In [ ]:
# =========================================================
# 0. Imports & Configuration
# =========================================================

# --- Standard library ---
import os
import sys
import glob
import importlib.util
import warnings
from datetime import datetime
from pathlib import Path

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"   # suppress MLflow progress bars

# --- Third-party core ---
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import stft

# --- scikit-learn ---
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- Other ML ---
import shap
import mlflow
from xgboost import XGBClassifier

# --- Deep learning (optional — 1D-CNN section guarded by HAS_TORCH) ---
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except ImportError:
    torch = None
    nn    = None
    HAS_TORCH = False
    print("PyTorch not installed — 1D-CNN sections will be skipped.")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# --- Project modules (all live in utils/) ---
_BASE_DIR = (
    Path(__file__).parent
    if "__file__" in dir()
    else Path(globals()["__vsc_ipynb_file__"]).parent
    if "__vsc_ipynb_file__" in globals()
    else Path().resolve()
)
# Auto-correct: VS Code CWD is often the workspace root, not the project subfolder.
# If utils/data_loader.py is not found here, step into bearing-fault-diagnosis/.
if not (_BASE_DIR / "utils" / "data_loader.py").exists():
    _BASE_DIR = _BASE_DIR / "bearing-fault-diagnosis"
assert (_BASE_DIR / "utils" / "data_loader.py").exists(), (
    "_BASE_DIR wrong: " + str(_BASE_DIR)
)
_UTILS    = _BASE_DIR / "utils"

def _load_module(name: str, path: Path, inject: bool = True):
    """Load a local .py module and optionally inject its public names here."""
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    if inject:
        _public = {k: v for k, v in vars(mod).items() if not k.startswith('_')}
        globals().update(_public)
    return mod

_load_module("download_dataset",  _UTILS / "download_dataset.py")
_load_module("data_loader",       _UTILS / "data_loader.py")
_load_module("dsp_features",      _UTILS / "dsp_features.py")
_load_module("ml_classification", _UTILS / "ml_classification.py")
_load_module("plot_style",        _UTILS / "plot_style.py")

import os, sys, warnings
warnings.filterwarnings("ignore")

print("utils/ loaded: download_dataset | data_loader | dsp_features | ml_classification | plot_style")

In [ ]:
# --- Constants (all tuneable values live here) ---
RANDOM_STATE = 42      # global seed
TEST_SIZE    = 0.20    # held-out test fraction
N_SPLITS     = 5       # cross-validation folds

# Signal-display
N_SHOW_SECONDS    = 0.1     # seconds of signal to display in time-domain plots

# Spectrum ranges  -  consistent across FFT and STFT for the same signal type
FFT_CURRENT_MAX_HZ    = 500
FFT_VIB_MAX_HZ        = 10000
STFT_CURRENT_MAX_HZ   = 500
STFT_VIB_MAX_HZ       = 10000
ENVELOPE_BAND         = (500, 10000)
ENVELOPE_CURRENT_BAND = (50, 5000)
ENVELOPE_MAX_HZ       = 300

# STFT
STFT_NPERSEG  = 2048
STFT_NOVERLAP = 1536

# Dataset selection
BEARING_SET = FULL_SET   # swap to MINIMAL_SET for real-damage-only (~2.4 GB)

# Operating condition filter
SETTING_FILTER  = None   #None for all conditions  'N15_M07_F10' 
USE_CURRENT     = True
USE_VIBRATION   = True

# DEV_MODE: cap signals loaded per bearing folder for fast iteration
N_SIGNALS_PER_BEARING = None

# Explainability
SHAP_SAMPLE_SIZE = 200

# --- Hyperparameter tuning iterations (RandomizedSearchCV n_iter per model) ---
GBT_N_ITER   = 30    # GBT: large continuous search space warrants more iterations
RF_N_ITER    = 30    # RF: similarly large space (n_estimators x max_features x depth)
XGB_N_ITER   = 30    # XGB: many regularisation knobs (alpha, lambda, colsample, etc.)

# --- 1D-CNN toggle & constants ---
RUN_CNN          = False  # set False to skip sections 5i-5j (requires GPU)
CNN_SEGMENT_LEN  = 8192
CNN_OVERLAP      = 0.50
CNN_EPOCHS       = 30
CNN_BATCH_SIZE   = 64
CNN_LR           = 1e-3

# --- MLflow ---
RUN_MLFLOW        = True                        # set False to skip all MLflow logging
MLFLOW_EXPERIMENT = "paderborn-bearing-fault"   # experiment name in the tracking UI

PLOTS_DIR = _BASE_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

_set_name  = 'MINIMAL_SET' if BEARING_SET is MINIMAL_SET else 'FULL_SET'
_mat_root  = _BASE_DIR / "paderborn_data" / "mat"
_on_disk   = sum(1 for b in BEARING_SET if (_mat_root / b).is_dir())

print(f"Pipeline initialised   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Base directory         {_BASE_DIR}")
print(f"Plots directory        {PLOTS_DIR}")
print(f"BEARING_SET            {_on_disk}/{len(BEARING_SET)} bearings on disk  ({_set_name})")
print(f"SETTING_FILTER         {SETTING_FILTER or 'ALL (4 conditions)'}")
print(f"N_SIGNALS_PER_BEARING  {N_SIGNALS_PER_BEARING}  (None = all)")
print(f"HAS_TORCH              {HAS_TORCH}")
print(f"RUN_CNN                {RUN_CNN}")
print(f"RUN_MLFLOW             {RUN_MLFLOW}")

## 1. Data Acquisition

`ensure_data` checks whether the required `.mat` files are already present and
downloads only the bearings listed in `MINIMAL_SET` if they are missing.
The returned path is the root directory that contains one sub-folder per bearing code.

In [ ]:
# =========================================================
# 1. Data Acquisition
# =========================================================

mat_dir = ensure_data(BEARING_SET)
print(f"\nData directory: {mat_dir}")

# Locate .mat files for a healthy bearing (K001) as the exploration example
mat_files = glob.glob(str(mat_dir / 'K001' / '*.mat'))

if not mat_files:
    raise FileNotFoundError(
        f"No .mat files found under {mat_dir / 'K001'}. "
        "Re-run ensure_data or check the download path."
    )

EXAMPLE_FILE = mat_files[0]
print(f"Example file  : {os.path.basename(EXAMPLE_FILE)}")
print(f"Files available: {len(mat_files)}")

## 2. Exploratory Data Analysis

Load one representative `.mat` file and inspect its metadata and signal statistics.
This confirms the data loaded correctly and surfaces the operating conditions
(speed, torque, force) that will become important for stratification later.
Bearing characteristic frequencies are calculated from the measured shaft speed
and will be used to interpret spectral plots in Section 7.

In [ ]:
# =========================================================
# 2. Exploratory Data Analysis
# =========================================================

sig = load_mat_file(EXAMPLE_FILE)

print(f"Bearing        : {sig.bearing_code}")
print(f"Condition      : {sig.label_name}  (class {sig.label_3class})")
print(f"Damage origin  : {sig.damage_origin}")
print(f"Operating setting: {sig.setting}")

print("\n--- Signal Info ---")
print(f"  Sampling rate : {sig.fs} Hz")
print(f"  Duration      : {sig.duration} s")
print(f"  Current 1   shape={sig.phase_current_1.shape}  "
      f"RMS={np.sqrt(np.mean(sig.phase_current_1**2)):.4f}")
print(f"  Current 2   shape={sig.phase_current_2.shape}  "
      f"RMS={np.sqrt(np.mean(sig.phase_current_2**2)):.4f}")
print(f"  Vibration   shape={sig.vibration.shape}  "
      f"RMS={np.sqrt(np.mean(sig.vibration**2)):.4f}")

print("\n--- Operating Conditions ---")
print(f"  Speed       : {sig.speed.mean():.1f} rpm")
print(f"  Torque      : {sig.torque.mean():.3f} Nm")
print(f"  Force       : {sig.force.mean():.1f} N")
print(f"  Temperature : {sig.temperature.mean():.1f} Â°C")

In [ ]:
# --- 2a. Bearing Characteristic Frequencies ---
# Derived from geometry; used as reference markers in spectral plots.

char_freqs = calc_characteristic_frequencies(sig.speed.mean())

print("Bearing characteristic frequencies:")
for name, f in char_freqs.items():
    print(f"  {name:6s}: {f:.2f} Hz")

## 3. Preprocessing Pipeline — DSP Feature Extraction

Raw time-series are transformed into a fixed-length feature vector per segment.
Four complementary feature groups capture different fault signatures:

| Group | Domain | Captures |
|---|---|---|
| Time-domain | Time | Amplitude statistics (RMS, kurtosis, crest factor …) |
| Frequency-domain | Frequency | Spectral centroid, bandwidth, harmonic ratios |
| WPD (Wavelet Packet) | Time-frequency | Sub-band energy distribution |
| Envelope | Frequency (demodulated) | Fault impulse repetition rates |

Each feature group is demonstrated on the example signal; the final cell runs
`extract_all_features` which combines all groups into one vector.

In [ ]:
# =========================================================
# 3. Preprocessing Pipeline
# =========================================================

# --- 3a. Time-Domain Features (Phase Current 1) ---
td = time_domain_features(sig.phase_current_1)
print("Time-domain features:")
for k, v in td.items():
    print(f"  {k:<25}: {v:.6f}")

In [ ]:
# --- 3b. Frequency-Domain Features ---
fd = frequency_domain_features(sig.phase_current_1, sig.fs)
print("Frequency-domain features:")
for k, v in fd.items():
    print(f"  {k:<25}: {v:.6f}")

In [ ]:
# --- 3c. Wavelet Packet Decomposition Features ---
wpd = wavelet_packet_features(sig.phase_current_1)
print(f"WPD features (first 8 of {len(wpd)}):")
for k, v in list(wpd.items())[:8]:
    print(f"  {k:<25}: {v:.6f}")

In [ ]:
# --- 3d. Envelope Analysis (Vibration) ---
# Bandpass → Hilbert → magnitude spectrum; fault frequencies appear as peaks.
env_freqs_vib, env_spec_vib = envelope_analysis(sig.vibration, sig.fs, band=ENVELOPE_BAND)

print("Envelope amplitudes at characteristic frequencies (vibration):")
for name, f_char in char_freqs.items():
    if f_char > 0:
        mask = (env_freqs_vib >= f_char - 2) & (env_freqs_vib <= f_char + 2)
        if np.any(mask):
            peak = np.max(env_spec_vib[mask])
            print(f"  {name:6s} ({f_char:.1f} Hz): {peak:.6f}")

In [ ]:
# --- 3e. Full Feature Vector ---
all_feats = extract_all_features(sig.phase_current_1, sig.fs, 'current', char_freqs)
print(f"Total features extracted per segment: {len(all_feats)}")

### 3f. Batch Feature Extraction

Load every bearing folder under `mat_dir`, optionally filtered to `SETTING_FILTER`.
When `SETTING_FILTER = None` all four operating conditions are used, giving
**~32 bearings × 4 conditions × 20 signals = ~2 560 samples**.

A two-stage condition-invariance pipeline is applied so that features are comparable
across operating conditions (speed / torque / radial load):

**Stage 1 — Signal-level normalisation (before feature extraction)**

Raw vibration and phase-current amplitudes vary systematically with operating condition.
Each signal is z-score normalised using per-condition mean / std computed from
**training bearings only** (same bearing split as Section 5a, same `RANDOM_STATE`):

```
sig_norm(t) = ( sig(t) − μ_cond ) / σ_cond
```

Because signal statistics are dominated by the carrier (shaft rotation), faults contribute
< 10 % of total signal energy — so the normalisation baseline is nearly fault-free and
generalises well to unseen bearings. Features extracted from the normalised signal
reflect fault severity rather than operating-point amplitude.

**Stage 2 — Frequency → shaft-order conversion (after feature extraction)**

Spectral features expressed in Hz scale with shaft speed. Dividing by f_shaft = rpm / 60
converts them to dimensionless shaft orders so the same fault pattern produces the same
feature value regardless of operating speed.

| Feature group | Raw units | After normalisation |
|---|---|---|
| `_spectral_centroid`, `_peak_frequency`, `_spectral_std` | Hz | shaft orders (÷ f_shaft) |
| `_spectral_variance` | Hz² | orders² (÷ f_shaft²) |
| Envelope (`env_*`), time-domain, WPD ratios | dimensionless | unchanged |

Results are cached to disk (keyed by feature-engineering parameters + source hash).
Runtime on a first run: 4–12 minutes depending on hardware.

In [ ]:
# --- 3f-i. Load all bearing signals (one operating condition) ---
# Iterate only BEARING_SET — never stray into extra folders that may exist on disk.
all_signals: list = []
for bearing_code in BEARING_SET:
    bearing_folder = mat_dir / bearing_code
    if not bearing_folder.is_dir():
        print(f"  WARNING: {bearing_code} not found on disk — skipping")
        continue
    sigs = load_dataset(str(bearing_folder), setting_filter=SETTING_FILTER)
    if N_SIGNALS_PER_BEARING is not None:
        sigs = sigs[:N_SIGNALS_PER_BEARING]
    all_signals.extend(sigs)

_actual_bearings = len(set(s.bearing_code for s in all_signals))
print(f"\nLoaded {len(all_signals)} signals  "
      f"({_actual_bearings}/{len(BEARING_SET)} bearings available, "
      f"up to {N_SIGNALS_PER_BEARING or 20} signals each)")
print("Class distribution:")
for label, name in enumerate(['Healthy', 'OR_damage', 'IR_damage']):
    n = sum(1 for s in all_signals if s.label_3class == label)
    print(f"  {label} -- {name}: {n}")

In [ ]:
# --- 3f-ii. Extract features -> build X, y, groups, conditions matrices ---
# Feature extraction is expensive. Results are cached to disk keyed by
# the parameters that affect the output. Re-run is triggered automatically when
# BEARING_SET, SETTING_FILTER, USE_CURRENT, USE_VIBRATION, N_SIGNALS_PER_BEARING,
# or the dsp_features.py source code changes.
# Cache version v3: adds per-condition signal-level normalisation before feature extraction.

import hashlib, pickle, dataclasses

# Include dsp_features.py content hash so any change to feature engineering
# automatically invalidates the cache — prevents stale features from being used.
_dsp_src_hash = hashlib.md5(
    (_BASE_DIR / "utils" / "dsp_features.py").read_bytes()
).hexdigest()[:8]

_cache_key = hashlib.md5(
    f"{sorted(BEARING_SET)}{SETTING_FILTER}{USE_CURRENT}{USE_VIBRATION}"
    f"{N_SIGNALS_PER_BEARING}{_dsp_src_hash}v3".encode()
).hexdigest()[:12]
_cache_path = _BASE_DIR / f".feature_cache_{_cache_key}.pkl"

# --- Pre-compute per-condition signal statistics from training bearings ---
# Signal amplitudes vary with operating condition (speed / load). Normalising
# each raw signal by its per-condition mean/std before feature extraction removes
# condition-induced amplitude differences while preserving fault-driven patterns.
# Uses the same StratifiedGroupKFold parameters as Section 5a -- since RANDOM_STATE
# and N_SPLITS are fixed, both splits are identical and consistent.
_labels_pre = np.array([s.label_3class for s in all_signals])
_groups_pre = np.array([s.bearing_code  for s in all_signals])
_sgkf_pre   = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
_tr_idx_pre, _ = next(_sgkf_pre.split(_labels_pre, _labels_pre, groups=_groups_pre))
_train_bearings_pre = set(_groups_pre[_tr_idx_pre])

_cond_signal_stats: dict = {}
for _cond in OPERATING_CONDITIONS:
    _sigs_c = [s for s in all_signals
               if s.setting == _cond and s.bearing_code in _train_bearings_pre]
    if _sigs_c:
        _vib  = np.concatenate([s.vibration       for s in _sigs_c])
        _cur1 = np.concatenate([s.phase_current_1 for s in _sigs_c])
        _cur2 = np.concatenate([s.phase_current_2 for s in _sigs_c])
        _cond_signal_stats[_cond] = {
            'vib_mean':  float(_vib.mean()),  'vib_std':  float(_vib.std()  + 1e-12),
            'cur1_mean': float(_cur1.mean()), 'cur1_std': float(_cur1.std() + 1e-12),
            'cur2_mean': float(_cur2.mean()), 'cur2_std': float(_cur2.std() + 1e-12),
        }
        print(f"  {_cond}: vib_std={_cond_signal_stats[_cond]['vib_std']:.5f}, "
              f"cur1_std={_cond_signal_stats[_cond]['cur1_std']:.5f}")
print(f"Signal stats from {len(_train_bearings_pre)} training bearings: "
      f"{sorted(_train_bearings_pre)}")


if _cache_path.exists():
    print(f"Loading cached features from {_cache_path.name} ...")
    with open(_cache_path, "rb") as _f:
        _cache = pickle.load(_f)
    X             = _cache["X"]
    y             = _cache["y"]
    groups        = _cache["groups"]
    conditions    = _cache["conditions"]
    feature_names = _cache["feature_names"]
    print(f"Loaded from cache: {X.shape}")

else:
    print("No cache found  -  computing features ...")

    def _normalize_freq_features(feats: dict, f_shaft: float) -> dict:
        """Convert absolute-frequency features to speed-invariant orders.

        Dividing Hz features by f_shaft (rpm/60) yields orders  -  multiples of
        the shaft rotation frequency  -  so the same fault pattern produces the same
        feature value regardless of operating speed.

        Args:
            feats:   Feature dictionary returned by extract_features_from_bearing.
            f_shaft: Shaft rotation frequency in Hz (= rpm / 60).

        Returns:
            Same dict with frequency features replaced by order-domain equivalents.
        """
        hz_suffixes = ('_spectral_centroid', '_spectral_std', '_peak_frequency')
        for k in list(feats):
            if any(k.endswith(s) for s in hz_suffixes):
                feats[k] = feats[k] / (f_shaft + 1e-10)
        for k in list(feats):
            if k.endswith('_spectral_variance'):
                feats[k] = feats[k] / (f_shaft**2 + 1e-10)
        return feats

    feature_list: list = []
    label_list:   list = []
    groups:       list = []
    conditions:   list = []   # operating condition string per sample

    for sig_i in tqdm(all_signals, desc="Extracting features", unit="signal"):
        rpm = OPERATING_CONDITIONS[sig_i.setting]['speed_rpm']
        char_freqs_i = calc_characteristic_frequencies(rpm)
        f_shaft_i    = rpm / 60

        # Normalise raw signal arrays by per-condition mean/std (from training bearings).
        # Creates a shallow copy of the dataclass — does not mutate the original signal.
        _st = _cond_signal_stats.get(sig_i.setting)
        if _st:
            sig_i = dataclasses.replace(
                sig_i,
                vibration       = (sig_i.vibration       - _st['vib_mean'])  / _st['vib_std'],
                phase_current_1 = (sig_i.phase_current_1 - _st['cur1_mean']) / _st['cur1_std'],
                phase_current_2 = (sig_i.phase_current_2 - _st['cur2_mean']) / _st['cur2_std'],
            )

        feats = extract_features_from_bearing(
            sig_i,
            use_current=USE_CURRENT,
            use_vibration=USE_VIBRATION,
            characteristic_freqs=char_freqs_i,
        )
        feats = _normalize_freq_features(feats, f_shaft_i)

        feature_list.append(list(feats.values()))
        label_list.append(sig_i.label_3class)
        groups.append(sig_i.bearing_code)
        conditions.append(sig_i.setting)

    feature_names = list(feats.keys())
    X          = np.array(feature_list, dtype=np.float32)
    y          = np.array(label_list,   dtype=np.int64)
    groups     = np.array(groups)
    conditions = np.array(conditions)

    with open(_cache_path, "wb") as _f:
        pickle.dump({"X": X, "y": y, "groups": groups,
                     "conditions": conditions,
                     "feature_names": feature_names}, _f)
    print(f"Features cached to {_cache_path.name}")

print(f"Feature matrix : {X.shape}  ({len(feature_names)} features per signal)")
print(f"Label vector   : {y.shape}  classes={np.bincount(y)}")
print(f"Unique bearings: {np.unique(groups)}")
print(f"Conditions     : {np.unique(conditions)}")

label_encoder = LabelEncoder().fit(y)
class_names   = ['Healthy', 'OR_damage', 'IR_damage']
display(
    pd.Series(y)
      .map(dict(enumerate(class_names)))
      .value_counts()
      .rename("count")
      .to_frame()
)

### 3g. DSP Signal Visualisations — Healthy vs Damaged

Each plot compares **K001 (Healthy)**, **KA04 (Outer Race Fault)**, and **KI04 (Inner Race Fault)**
under identical operating conditions. The three-way layout highlights how OR and IR faults differ
in envelope spectral content — the key to resolving the OR/IR confusion seen in the confusion matrix.

In [ ]:
# --- 3h. Load damaged bearing example (KA04 — Outer Race Fault) ---
# KA04 is a real fatigue-induced outer race damage bearing from MINIMAL_SET.
# When SETTING_FILTER is None, pick any available file; otherwise match the filter.
_dmg_pattern = (
    str(mat_dir / 'KA04' / f'*{SETTING_FILTER}*.mat')
    if SETTING_FILTER
    else str(mat_dir / 'KA04' / '*.mat')
)
_dmg_files = glob.glob(_dmg_pattern)
if not _dmg_files:
    raise FileNotFoundError(
        f"No KA04 files found for setting {SETTING_FILTER}. "
        "Check mat_dir or run ensure_data."
    )

sig_dmg = load_mat_file(_dmg_files[0])
char_freqs_dmg = calc_characteristic_frequencies(sig_dmg.speed.mean())

print(f"Damaged bearing  : {sig_dmg.bearing_code}")
print(f"Condition        : {sig_dmg.label_name}  (class {sig_dmg.label_3class})")
print(f"Damage origin    : {sig_dmg.damage_origin}")
print(f"Operating setting: {sig_dmg.setting}")
print(f"Speed            : {sig_dmg.speed.mean():.1f} rpm")
print("\nCharacteristic frequencies:")
for name, f in char_freqs_dmg.items():
    print(f"  {name:6s}: {f:.2f} Hz")

In [ ]:
# --- 3h-ii. Load inner race damaged bearing example (KI04) ---
# KI04 is a real fatigue-induced inner race damage bearing — used as the IR_damage
# reference signal in all five DSP comparison plots that follow.
_ir_pattern = (
    str(mat_dir / 'KI04' / f'*{SETTING_FILTER}*.mat')
    if SETTING_FILTER
    else str(mat_dir / 'KI04' / '*.mat')
)
_ir_files = glob.glob(_ir_pattern)
if not _ir_files:
    raise FileNotFoundError(
        f"No KI04 files found for setting {SETTING_FILTER}. "
        "Check mat_dir or run ensure_data."
    )

sig_ir = load_mat_file(_ir_files[0])
char_freqs_ir = calc_characteristic_frequencies(sig_ir.speed.mean())

print(f"IR damaged bearing : {sig_ir.bearing_code}")
print(f"Condition          : {sig_ir.label_name}  (class {sig_ir.label_3class})")
print(f"Damage origin      : {sig_ir.damage_origin}")
print(f"Operating setting  : {sig_ir.setting}")
print(f"Speed              : {sig_ir.speed.mean():.1f} rpm")
print("\nCharacteristic frequencies:")
for name, f in char_freqs_ir.items():
    print(f"  {name:6s}: {f:.2f} Hz")

In [ ]:
# --- 3g-i. Time-Domain Signals Comparison (Healthy / OR_damage / IR_damage) ---
_n_show = int(N_SHOW_SECONDS * sig.fs)
_t_ms   = sig.time_64k[:_n_show] * 1000   # ms

_rows = [
    (sig.phase_current_1, sig_dmg.phase_current_1, sig_ir.phase_current_1, "Current 1 [A]", C1, D1, I1),
    (sig.phase_current_2, sig_dmg.phase_current_2, sig_ir.phase_current_2, "Current 2 [A]", C2, D2, I2),
    (sig.vibration,       sig_dmg.vibration,       sig_ir.vibration,       "Vibration [g]", C3, D3, I3),
]

fig, axes = plt.subplots(3, 3, figsize=(13, 6.5), sharex=True)
axes[0, 0].set_title(f"Healthy — {sig.bearing_code} ({sig.setting})", fontsize=9)
axes[0, 1].set_title(f"OR_damage — {sig_dmg.bearing_code} ({sig_dmg.label_name})", fontsize=9)
axes[0, 2].set_title(f"IR_damage — {sig_ir.bearing_code} ({sig_ir.label_name})", fontsize=9)

for row, (h_sig, d_sig, i_sig, ylabel, h_col, d_col, i_col) in enumerate(_rows):
    axes[row, 0].plot(_t_ms, h_sig[:_n_show], lw=0.5, color=h_col)
    axes[row, 1].plot(_t_ms, d_sig[:_n_show], lw=0.5, color=d_col)
    axes[row, 2].plot(_t_ms, i_sig[:_n_show], lw=0.5, color=i_col)
    axes[row, 0].set_ylabel(ylabel, fontsize=8)

for col in range(3):
    axes[2, col].set_xlabel("Time [ms]")

fig.suptitle("Time-Domain Signals Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_time_domain_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-ii. FFT Spectrum Comparison (Healthy / OR_damage / IR_damage) ---
def _fft_db(signal, fs):
    N = len(signal)
    freqs = np.fft.fftfreq(N, 1 / fs)[: N // 2]
    mag   = np.abs(np.fft.fft(signal))[: N // 2] * 2 / N
    return freqs, 20 * np.log10(mag + 1e-10)

_fc_h, _mc_h = _fft_db(sig.phase_current_1,    sig.fs)
_fv_h, _mv_h = _fft_db(sig.vibration,           sig.fs)
_fc_d, _mc_d = _fft_db(sig_dmg.phase_current_1, sig_dmg.fs)
_fv_d, _mv_d = _fft_db(sig_dmg.vibration,       sig_dmg.fs)
_fc_i, _mc_i = _fft_db(sig_ir.phase_current_1,  sig_ir.fs)
_fv_i, _mv_i = _fft_db(sig_ir.vibration,        sig_ir.fs)

fig, axes = plt.subplots(2, 3, figsize=(13, 5), sharey='row')
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"OR_damage — {sig_dmg.bearing_code}", fontsize=9)
axes[0, 2].set_title(f"IR_damage — {sig_ir.bearing_code}", fontsize=9)

# Row 0: Phase Current 1
for ax, freqs, mag, color, vcolor in [
    (axes[0, 0], _fc_h, _mc_h, C1, C3),
    (axes[0, 1], _fc_d, _mc_d, D1, D3),
    (axes[0, 2], _fc_i, _mc_i, I1, I3),
]:
    mask = freqs <= FFT_CURRENT_MAX_HZ
    ax.plot(freqs[mask], mag[mask], lw=0.5, color=color)
    ax.axvline(x=100, color=vcolor, ls='--', alpha=0.6, label='100 Hz')
    ax.legend(fontsize=7)

# Row 1: Vibration
for ax, freqs, mag, color, cf, fc in [
    (axes[1, 0], _fv_h, _mv_h, C2, char_freqs,    FAULT_COLORS),
    (axes[1, 1], _fv_d, _mv_d, D2, char_freqs_dmg, FAULT_COLORS_DMG),
    (axes[1, 2], _fv_i, _mv_i, I2, char_freqs_ir,  FAULT_COLORS_IR),
]:
    mask = freqs <= FFT_VIB_MAX_HZ
    ax.plot(freqs[mask], mag[mask], lw=0.5, color=color)
    for name, f in cf.items():
        if name in fc:
            ax.axvline(x=f, color=fc[name], ls='--', alpha=0.6, label=name)
    ax.legend(fontsize=6, ncol=2)

axes[0, 0].set_ylabel("Phase Current 1 [dB]", fontsize=8)
axes[1, 0].set_ylabel("Vibration [dB]",        fontsize=8)
for col in range(3):
    axes[1, col].set_xlabel("Frequency [Hz]")

fig.suptitle("FFT Spectrum Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_fft_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-iii. STFT Spectrogram Comparison (Healthy / OR_damage / IR_damage) ---
# Use a fixed-duration segment so STFT and CWT use the same time window.
_STFT_DUR_S = 1.0
_n_stft = int(_STFT_DUR_S * sig.fs)

_c1_h_stft  = sig.phase_current_1[:_n_stft]
_vib_h_stft = sig.vibration[:_n_stft]
_c1_d_stft  = sig_dmg.phase_current_1[:_n_stft]
_vib_d_stft = sig_dmg.vibration[:_n_stft]
_c1_i_stft  = sig_ir.phase_current_1[:_n_stft]
_vib_i_stft = sig_ir.vibration[:_n_stft]

_f_s, _t_s, _Zch = stft(_c1_h_stft,  fs=sig.fs,     nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP)
_f_v, _t_v, _Zvh = stft(_vib_h_stft, fs=sig.fs,     nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP)
_,    _,    _Zcd  = stft(_c1_d_stft,  fs=sig_dmg.fs, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP)
_,    _,    _Zvd  = stft(_vib_d_stft, fs=sig_dmg.fs, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP)
_,    _,    _Zci  = stft(_c1_i_stft,  fs=sig_ir.fs,  nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP)
_,    _,    _Zvi  = stft(_vib_i_stft, fs=sig_ir.fs,  nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP)

_mf_c = _f_s <= STFT_CURRENT_MAX_HZ
_mf_v = _f_v <= STFT_VIB_MAX_HZ

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"OR_damage — {sig_dmg.bearing_code}", fontsize=9)
axes[0, 2].set_title(f"IR_damage — {sig_ir.bearing_code}", fontsize=9)

for (ax, t, f, f_mask, Z, cmap) in [
    (axes[0, 0], _t_s, _f_s, _mf_c, _Zch, CMAP),
    (axes[0, 1], _t_s, _f_s, _mf_c, _Zcd, CMAP_DMG),
    (axes[0, 2], _t_s, _f_s, _mf_c, _Zci, CMAP_IR),
    (axes[1, 0], _t_v, _f_v, _mf_v, _Zvh, CMAP),
    (axes[1, 1], _t_v, _f_v, _mf_v, _Zvd, CMAP_DMG),
    (axes[1, 2], _t_v, _f_v, _mf_v, _Zvi, CMAP_IR),
]:
    C = 20 * np.log10(np.abs(Z[f_mask]) + 1e-10)
    t_plot = t[:C.shape[1]]
    pcm = ax.pcolormesh(t_plot, f[f_mask], C, cmap=cmap, shading='gouraud')
    fig.colorbar(pcm, ax=ax, label='dB', pad=0.02)
    ax.set_xlim(0, _STFT_DUR_S)

axes[0, 0].set_ylabel("Phase Current\nFrequency [Hz]", fontsize=8)
axes[1, 0].set_ylabel("Vibration\nFrequency [Hz]", fontsize=8)
for col in range(3):
    axes[1, col].set_xlabel("Time [s]")

fig.suptitle("STFT Spectrogram Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_stft_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-iv. Envelope Spectrum Comparison (Healthy / OR_damage / IR_damage) ---
# Envelope peaks at BPFO/BPFI harmonics are the primary discriminator between
# outer-race and inner-race faults — this plot makes that difference explicit.
_efv_h, _esv_h = env_freqs_vib, env_spec_vib   # healthy vibration (from sec3-envelope)
_efv_d, _esv_d = envelope_analysis(sig_dmg.vibration,       sig_dmg.fs, band=ENVELOPE_BAND)
_efv_i, _esv_i = envelope_analysis(sig_ir.vibration,        sig_ir.fs,  band=ENVELOPE_BAND)
_efc_h, _esc_h = envelope_analysis(sig.phase_current_1,     sig.fs,     band=ENVELOPE_CURRENT_BAND)
_efc_d, _esc_d = envelope_analysis(sig_dmg.phase_current_1, sig_dmg.fs, band=ENVELOPE_CURRENT_BAND)
_efc_i, _esc_i = envelope_analysis(sig_ir.phase_current_1,  sig_ir.fs,  band=ENVELOPE_CURRENT_BAND)

def _env_axes(ax, freqs, spec, cf_dict, fc_dict, color):
    mask = freqs <= ENVELOPE_MAX_HZ
    ax.plot(freqs[mask], 20 * np.log10(spec[mask] + 1e-10), lw=0.8, color=color)
    for name, f in cf_dict.items():
        if name in fc_dict and f <= ENVELOPE_MAX_HZ:
            ax.axvline(x=f, color=fc_dict[name], ls='--', alpha=0.7, label=name)
            for h in [2, 3]:
                if f * h <= ENVELOPE_MAX_HZ:
                    ax.axvline(x=f * h, color=fc_dict[name], ls=':', alpha=0.3)
    ax.legend(fontsize=6, ncol=2)

fig, axes = plt.subplots(2, 3, figsize=(13, 5))
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"OR_damage — {sig_dmg.bearing_code}", fontsize=9)
axes[0, 2].set_title(f"IR_damage — {sig_ir.bearing_code}", fontsize=9)

_env_axes(axes[0, 0], _efv_h, _esv_h, char_freqs,    FAULT_COLORS,    C2)
_env_axes(axes[0, 1], _efv_d, _esv_d, char_freqs_dmg, FAULT_COLORS_DMG, D2)
_env_axes(axes[0, 2], _efv_i, _esv_i, char_freqs_ir,  FAULT_COLORS_IR,  I2)
_env_axes(axes[1, 0], _efc_h, _esc_h, char_freqs,    FAULT_COLORS,    C1)
_env_axes(axes[1, 1], _efc_d, _esc_d, char_freqs_dmg, FAULT_COLORS_DMG, D1)
_env_axes(axes[1, 2], _efc_i, _esc_i, char_freqs_ir,  FAULT_COLORS_IR,  I1)

axes[0, 0].set_ylabel("Vibration [dB]",       fontsize=8)
axes[1, 0].set_ylabel("Phase Current 1 [dB]", fontsize=8)
for col in range(3):
    axes[1, col].set_xlabel("Frequency [Hz]")

fig.suptitle("Envelope Spectrum Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_envelope_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-v. CWT Scalogram Comparison (Healthy / OR_damage / IR_damage) ---
# Complex Morlet CWT: adaptive time-frequency resolution — captures impulsive
# fault signatures at different frequency scales for all three bearing states.
_CWT_WAVELET  = 'cmor1.5-1.0'
_CWT_DS       = 8        # downsample 64 kHz → 8 kHz
_CWT_DUR_S    = 1.0
_CWT_N_SCALES = 100
_fs_cwt = sig.fs / _CWT_DS
_dt_cwt = 1.0 / _fs_cwt
_n_cwt  = int(_CWT_DUR_S * _fs_cwt)

_c1_h  = sig.phase_current_1    [: _n_cwt * _CWT_DS : _CWT_DS]
_vib_h = sig.vibration           [: _n_cwt * _CWT_DS : _CWT_DS]
_c1_d  = sig_dmg.phase_current_1 [: _n_cwt * _CWT_DS : _CWT_DS]
_vib_d = sig_dmg.vibration       [: _n_cwt * _CWT_DS : _CWT_DS]
_c1_i  = sig_ir.phase_current_1  [: _n_cwt * _CWT_DS : _CWT_DS]
_vib_i = sig_ir.vibration        [: _n_cwt * _CWT_DS : _CWT_DS]
_t_cwt = np.arange(_n_cwt) * _dt_cwt

_f_c_max, _f_v_max, _f_min_cwt = 500, 3000, 5
_sc_c = np.logspace(np.log10(_fs_cwt / _f_c_max),
                    np.log10(_fs_cwt / _f_min_cwt), _CWT_N_SCALES)
_sc_v = np.logspace(np.log10(_fs_cwt / _f_v_max),
                    np.log10(_fs_cwt / _f_min_cwt), _CWT_N_SCALES)

_cc_h, _fq_c = pywt.cwt(_c1_h,  _sc_c, _CWT_WAVELET, sampling_period=_dt_cwt)
_cv_h, _fq_v = pywt.cwt(_vib_h, _sc_v, _CWT_WAVELET, sampling_period=_dt_cwt)
_cc_d, _      = pywt.cwt(_c1_d,  _sc_c, _CWT_WAVELET, sampling_period=_dt_cwt)
_cv_d, _      = pywt.cwt(_vib_d, _sc_v, _CWT_WAVELET, sampling_period=_dt_cwt)
_cc_i, _      = pywt.cwt(_c1_i,  _sc_c, _CWT_WAVELET, sampling_period=_dt_cwt)
_cv_i, _      = pywt.cwt(_vib_i, _sc_v, _CWT_WAVELET, sampling_period=_dt_cwt)

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"OR_damage — {sig_dmg.bearing_code}", fontsize=9)
axes[0, 2].set_title(f"IR_damage — {sig_ir.bearing_code}", fontsize=9)

for (ax, coefs, cmap, ylabel) in [
    (axes[0, 0], _cc_h, CMAP,     "Phase Current\nFrequency [Hz]"),
    (axes[0, 1], _cc_d, CMAP_DMG, None),
    (axes[0, 2], _cc_i, CMAP_IR,  None),
    (axes[1, 0], _cv_h, CMAP,     "Vibration\nFrequency [Hz]"),
    (axes[1, 1], _cv_d, CMAP_DMG, None),
    (axes[1, 2], _cv_i, CMAP_IR,  None),
]:
    freqs = _fq_c if ax in (axes[0, 0], axes[0, 1], axes[0, 2]) else _fq_v
    pcm = ax.pcolormesh(_t_cwt, freqs, np.abs(coefs), cmap=cmap, shading='gouraud')
    fig.colorbar(pcm, ax=ax, label='Magnitude', pad=0.02)
    ax.set_yscale('log')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=8)

for col in range(3):
    axes[1, col].set_xlabel("Time [s]")

fig.suptitle("CWT Scalogram Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_cwt_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Model Definition

`TraditionalMLPipeline` (from `03_ml_classification.py`) wraps three classifiers —
Random Forest, **Gradient Boosting** (tuned in Section 5c), and XGBoost —
plus a **Stacking meta-ensemble** (RF + GBT + XGB → LogisticRegression) added in Section 5f.

A **1D-CNN** trained directly on raw vibration segments is defined in Section 5h–5i
as a deep-learning comparison (requires GPU for practical training times).

In [ ]:
# =========================================================
# 4. Model Definition
# =========================================================

pipeline_runner = TraditionalMLPipeline()

print("Models registered:")
for name in pipeline_runner.pipelines:
    print(f"  {name}")

## 5. Cross-Validation & Model Selection

### Data leakage controls

Two sources of leakage are explicitly prevented:

1. **Bearing-level leakage** — `StratifiedGroupKFold` keeps all signals from the same
   physical bearing in the same fold. Without this, a model can memorise bearing-specific
   vibration signatures and report inflated CV scores (~1.0) that do not generalise to
   unseen bearings.

2. **Preprocessing leakage** — each classifier is wrapped in a
   `sklearn.pipeline.Pipeline([("scaler", StandardScaler()), ("model", clf)])`.
   The scaler is re-fit inside every CV fold so no test-fold statistics contaminate
   the scaler fit.

### Steps

| Step | Technique | Benefit |
|---|---|---|
| **5a** | `StratifiedGroupKFold` train / test split (bearing-aware) | Held-out bearings never seen during training or tuning |
| **5b** | `SelectFromModel` (RF-based, median threshold) | Reduces 183 → ~92 features; removes noise dimensions |
| **5c** | `RandomizedSearchCV` on GBT (30 iters, 3-fold inner CV) | Tunes learning rate, depth, subsample, regularisation |
| **5d** | `RandomizedSearchCV` on RF (30 iters, 3-fold inner CV) | Tunes n_estimators, max_features, min_samples_leaf |
| **5e** | `RandomizedSearchCV` on XGB (30 iters, 3-fold inner CV) | Tunes learning rate, depth, L1/L2 regularisation, colsample |
| **5f** | Stacking meta-ensemble (RF + GBT + XGB → LogisticRegression) | Combines model diversity |
| **5g** | Train all models on the feature-selected matrix | Final fit before test evaluation |
| **5h–5i** | 1D-CNN on raw vibration segments | Deep-learning comparison baseline |

The test set remains **invisible** throughout steps 5b–5g — feature selection and
hyperparameter tuning operate exclusively on `X_train_fs`.

In [ ]:
# =========================================================
# 5. Cross-Validation & Model Selection
# =========================================================

# --- 5a. Train / Test Split ---
# StratifiedGroupKFold guarantees two things simultaneously:
#   1. All signals from the same bearing stay in the same fold (no leakage)
#   2. Each fold has approximately the same class distribution
# With 5 bearings per class and n_splits=5, each test fold gets exactly
# 1 bearing per class (20 signals/class → 60 signals total in test).
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Propagate condition labels alongside the feature split
conditions_train = conditions[train_idx]
conditions_test  = conditions[test_idx]

train_bearings = np.unique(groups[train_idx])
test_bearings  = np.unique(groups[test_idx])
print(f"Train: {X_train.shape}  bearings={list(train_bearings)}")
print(f"Test : {X_test.shape}   bearings={list(test_bearings)}")
print(f"Class distribution — train: {np.bincount(y_train)}  test: {np.bincount(y_test)}")

In [ ]:
# --- 5b. Feature Selection ---
# 171 features from ~500 training samples is a high-dimensional regime.
# Noise features add variance without improving signal — particularly harmful for
# tree ensembles where irrelevant splits waste capacity.
# A quick RandomForest (scale-invariant) is fitted on X_train to rank importances;
# SelectFromModel keeps only those above the median threshold (~85 features).
_fs_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
_fs_rf.fit(X_train, y_train)   # fit on raw (unscaled) X_train — RF is scale-invariant

selector = SelectFromModel(_fs_rf, threshold='median', prefit=True)

X_train_fs = selector.transform(X_train)
X_test_fs  = selector.transform(X_test)
feature_names_fs = [feature_names[i] for i in selector.get_support(indices=True)]

print(f"Feature selection: {X_train.shape[1]} → {X_train_fs.shape[1]} features kept")
print(f"Removed           : {X_train.shape[1] - X_train_fs.shape[1]} low-importance features")

_imp = _fs_rf.feature_importances_[selector.get_support(indices=True)]
print("\nTop 15 retained features:")
for _name, _i in sorted(zip(feature_names_fs, _imp), key=lambda x: -x[1])[:15]:
    print(f"  {_name:<50} {_i:.4f}")

In [ ]:
# --- 5c. GBT Hyperparameter Tuning ---
# RandomizedSearchCV on X_train_fs (feature-selected) with 3-fold inner CV.
# _gbt_pipe is a local search pipeline; the tuned params are written back into
# pipeline_runner.pipelines['GBT'] via set_params so the scaler stays inside.
#
# Conservative search space: shallow trees (depth 3-4), slow learning rates (<=0.1),
# and subsample regularisation prevent the tuner from finding params that look good
# in-fold but overfit on a small dataset (~480 samples, ~85 features).
_gbt_pipe = SKPipeline([
    ('scaler', StandardScaler()),
    ('gbt',    GradientBoostingClassifier(random_state=RANDOM_STATE)),
])
_param_dist = {
    'gbt__n_estimators':      [100, 200, 300],
    'gbt__learning_rate':     [0.01, 0.05, 0.1],
    'gbt__max_depth':         [3, 4],
    'gbt__min_samples_split': [5, 10, 20],
    'gbt__min_samples_leaf':  [3, 5, 10],
    'gbt__subsample':         [0.7, 0.8, 0.9],
    'gbt__max_features':      ['sqrt', 0.7, 0.8],
}
# StratifiedGroupKFold ensures no bearing appears in both train and val fold,
# preventing the inflated CV scores caused by memorising bearing-specific patterns.
_cv_inner   = StratifiedGroupKFold(n_splits=3)
_gbt_search = RandomizedSearchCV(
    _gbt_pipe, _param_dist,
    n_iter=GBT_N_ITER, cv=_cv_inner,
    scoring='f1_macro', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
_gbt_search.fit(X_train_fs, y_train, groups=groups[train_idx])

_best = _gbt_search.best_params_
print(f"Best GBT params : { {k.replace('gbt__', ''): v for k, v in _best.items()} }")
print(f"Best CV F1-macro: {_gbt_search.best_score_:.4f}")

pipeline_runner.pipelines['GBT'].set_params(model=GradientBoostingClassifier(
    n_estimators      = _best['gbt__n_estimators'],
    learning_rate     = _best['gbt__learning_rate'],
    max_depth         = _best['gbt__max_depth'],
    min_samples_split = _best['gbt__min_samples_split'],
    min_samples_leaf  = _best['gbt__min_samples_leaf'],
    subsample         = _best['gbt__subsample'],
    max_features      = _best['gbt__max_features'],
    random_state      = RANDOM_STATE,
))
print("pipeline_runner.pipelines['GBT'] updated with tuned configuration.")

In [ ]:
# --- 5d. RF Hyperparameter Tuning ---
# Random Forest is the current best model (baseline ~77 %).
# Tuning focuses on the three knobs that matter most on this dataset:
#   - n_estimators: more trees reduce variance (diminishing returns after ~300)
#   - max_features: controls diversity between trees — 'sqrt' is the RF default,
#     but lower fractions can help when features are highly correlated
#   - min_samples_leaf: primary regularisation against overfitting on small datasets
# max_depth is left unconstrained (None) — bagging already controls variance.
_rf_pipe = SKPipeline([
    ('scaler', StandardScaler()),
    ('rf',     RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])
_rf_param_dist = {
    'rf__n_estimators':      [200, 300, 500],
    'rf__max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf':  [1, 2, 4],
    'rf__max_depth':         [None, 20, 30],
    'rf__class_weight':      [None, 'balanced'],
}
_cv_inner = StratifiedGroupKFold(n_splits=3)
_rf_search = RandomizedSearchCV(
    _rf_pipe, _rf_param_dist,
    n_iter=RF_N_ITER, cv=_cv_inner,
    scoring='f1_macro', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
_rf_search.fit(X_train_fs, y_train, groups=groups[train_idx])

_best_rf = _rf_search.best_params_
print(f"Best RF params  : { {k.replace('rf__', ''): v for k, v in _best_rf.items()} }")
print(f"Best CV F1-macro: {_rf_search.best_score_:.4f}")

pipeline_runner.pipelines['RF'].set_params(model=RandomForestClassifier(
    n_estimators      = _best_rf['rf__n_estimators'],
    max_features      = _best_rf['rf__max_features'],
    min_samples_split = _best_rf['rf__min_samples_split'],
    min_samples_leaf  = _best_rf['rf__min_samples_leaf'],
    max_depth         = _best_rf['rf__max_depth'],
    class_weight      = _best_rf['rf__class_weight'],
    random_state      = RANDOM_STATE,
    n_jobs            = -1,
))
print("pipeline_runner.pipelines['RF'] updated with tuned configuration.")

In [ ]:
# --- 5e. XGB Hyperparameter Tuning ---
# XGBoost exposes more regularisation knobs than GBT.
# Key levers on a small dataset (~500 samples, ~85 features):
#   - learning_rate x n_estimators: shrinkage controls overfitting
#   - subsample / colsample_bytree: stochastic sampling adds noise robustness
#   - reg_alpha (L1) / reg_lambda (L2): penalise complex trees directly
#   - min_child_weight: minimum sum of instance weights in a leaf (prevents tiny splits)
_xgb_pipe = SKPipeline([
    ('scaler', StandardScaler()),
    ('xgb',    XGBClassifier(random_state=RANDOM_STATE,
                              eval_metric='mlogloss', verbosity=0)),
])
_xgb_param_dist = {
    'xgb__n_estimators':     [100, 200, 300, 500],
    'xgb__learning_rate':    [0.01, 0.05, 0.1],
    'xgb__max_depth':        [3, 4, 5, 6],
    'xgb__subsample':        [0.7, 0.8, 0.9],
    'xgb__colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'xgb__reg_alpha':        [0, 0.1, 0.5, 1.0],
    'xgb__reg_lambda':       [1.0, 2.0, 5.0],
    'xgb__min_child_weight': [1, 3, 5],
}
_cv_inner = StratifiedGroupKFold(n_splits=3)
_xgb_search = RandomizedSearchCV(
    _xgb_pipe, _xgb_param_dist,
    n_iter=XGB_N_ITER, cv=_cv_inner,
    scoring='f1_macro', n_jobs=-1,
    random_state=RANDOM_STATE, verbose=0,
)
_xgb_search.fit(X_train_fs, y_train, groups=groups[train_idx])

_best_xgb = _xgb_search.best_params_
print(f"Best XGB params : { {k.replace('xgb__', ''): v for k, v in _best_xgb.items()} }")
print(f"Best CV F1-macro: {_xgb_search.best_score_:.4f}")

pipeline_runner.pipelines['XGB'].set_params(model=XGBClassifier(
    n_estimators     = _best_xgb['xgb__n_estimators'],
    learning_rate    = _best_xgb['xgb__learning_rate'],
    max_depth        = _best_xgb['xgb__max_depth'],
    subsample        = _best_xgb['xgb__subsample'],
    colsample_bytree = _best_xgb['xgb__colsample_bytree'],
    reg_alpha        = _best_xgb['xgb__reg_alpha'],
    reg_lambda       = _best_xgb['xgb__reg_lambda'],
    min_child_weight = _best_xgb['xgb__min_child_weight'],
    random_state     = RANDOM_STATE,
    eval_metric      = 'mlogloss',
    verbosity        = 0,
))
print("pipeline_runner.pipelines['XGB'] updated with tuned configuration.")

In [ ]:
# --- 5f. Stacking Classifier Setup ---
# StackingClassifier trains a meta-learner on out-of-fold predictions from the
# three base models, using 5-fold cross-validation internally.
#
# Why this helps the OR/IR confusion:
#   The base models already achieve ~90% on Healthy detection but disagree on
#   OR vs IR boundary cases. The meta-learner (LogisticRegression) learns a
#   weighted decision boundary specifically in the space of base model outputs —
#   it can learn "when RF says OR but XGB says IR, trust XGB".
#
# Each base estimator is passed as the full Pipeline (scaler + model) so that
# StackingClassifier re-fits the scaler inside each CV fold — no leakage.
# The Stacking pipeline has no extra outer scaler because the meta-learner
# receives probabilities in [0, 1], which need no further scaling.
pipeline_runner.pipelines['Stacking'] = SKPipeline([
    ('model', StackingClassifier(
        estimators=[
            ('RF',  pipeline_runner.pipelines['RF']),
            ('GBT', pipeline_runner.pipelines['GBT']),
            ('XGB', pipeline_runner.pipelines['XGB']),
        ],
        final_estimator=LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
        ),
        cv=5,               # 5-fold CV to generate out-of-fold meta-features
        passthrough=False,  # meta-learner sees only predicted probas, not raw features
        n_jobs=-1,
    )),
])
print("Stacking model added: RF + GBT + XGB -> LogisticRegression meta-learner")

In [ ]:
# --- 5g. Train All Models ---
# All classifiers receive the feature-selected matrix (signal already normalised
# per condition in Section 3f before feature extraction).
# Steps 5c–5e replaced each model's default parameters with tuned configurations.
# Step 5f added the Stacking meta-ensemble on top of the three tuned base models.
results = pipeline_runner.train_and_evaluate(
    X_train_fs, y_train, X_test_fs, y_test,
    feature_names=feature_names_fs,
)

In [ ]:
# --- 5h. 1D-CNN Data Preparation ---
if not (RUN_CNN and HAS_TORCH):
    print("1D-CNN skipped (RUN_CNN=False or PyTorch not installed).")
else:
    # Reuse the bearing-level train/test indices from 5a so the CNN sees the same
    # held-out bearings as the traditional ML models — a fair apples-to-apples comparison.
    # Signals have slightly variable lengths across recordings, so we process each one
    # individually rather than stacking, then concatenate the resulting segments.

    def _segment_signals(indices, overlap: float):
        """Segment raw vibration signals for a given index set.

        Args:
            indices: Array of signal indices into all_signals.
            overlap: Fractional overlap between consecutive segments.

        Returns:
            Tuple of (segments array (N, L), labels array (N,))
        """
        seg_list, lbl_list = [], []
        for i in indices:
            _sig = all_signals[i].vibration.astype(np.float32)[np.newaxis]  # (1, L)
            _segs, _lbls = prepare_segments(_sig, np.array([y[i]]), CNN_SEGMENT_LEN, overlap)
            seg_list.append(_segs)
            lbl_list.append(_lbls)
        return np.concatenate(seg_list), np.concatenate(lbl_list)

    # Training: overlapping windows maximise diversity of segments per bearing
    X_train_seg, y_train_seg = _segment_signals(train_idx, CNN_OVERLAP)

    # Inference: non-overlapping windows; majority-vote aggregation in step 5i
    X_test_seg,  y_test_seg  = _segment_signals(test_idx,  overlap=0.0)

    X_train_normn = prepare_1d_cnn_data(X_train_seg)   # (N, 1, L)
    X_test_normn  = prepare_1d_cnn_data(X_test_seg)

    print(f"Train segments : {X_train_normn.shape}  ({X_train_normn.nbytes / 1e6:.0f} MB)")
    print(f"Test  segments : {X_test_normn.shape}  ({X_test_normn.nbytes / 1e6:.0f} MB)")
    print(f"Approx. segments per test signal : "
          f"{X_test_normn.shape[0] // len(test_idx)}")

In [ ]:
# --- 5i. 1D-CNN Training ---
if not (RUN_CNN and HAS_TORCH):
    print("1D-CNN skipped (RUN_CNN=False or PyTorch not installed).")
else:
    # The CNN learns directly from raw vibration waveforms without any hand-crafted features.
    # Overlapping windows during training act as data augmentation;
    # no overlap is used at inference to avoid within-signal correlation.
    cnn_model = build_1d_cnn_model(input_length=CNN_SEGMENT_LEN, n_classes=3)

    # Hold back the last 20% of training segments for validation monitoring.
    # Contiguous split preserves temporal structure within each bearing.
    _n_val = int(0.2 * len(X_train_seg))
    _X_tr, _X_val = X_train_normn[:-_n_val], X_train_normn[-_n_val:]
    _y_tr, _y_val = y_train_seg[:-_n_val], y_train_seg[-_n_val:]

    cnn_model, cnn_history = train_pytorch_model(
        cnn_model, _X_tr, _y_tr, _X_val, _y_val,
        epochs=CNN_EPOCHS, batch_size=CNN_BATCH_SIZE, lr=CNN_LR,
    )

    # Segment-level predictions -> signal-level majority vote
    cnn_model.eval()
    _device = next(cnn_model.parameters()).device
    with torch.no_grad():
        _logits    = cnn_model(torch.FloatTensor(X_test_normn).to(_device))
        _seg_preds = _logits.argmax(dim=1).cpu().numpy()

    # Reconstruct how many non-overlapping segments each test signal produced
    _segs_per_sig = [
        (len(all_signals[i].vibration) - CNN_SEGMENT_LEN) // CNN_SEGMENT_LEN + 1
        for i in test_idx
    ]
    _offset = 0
    y_cnn_pred = []
    for _n in _segs_per_sig:
        _votes = _seg_preds[_offset: _offset + _n]
        y_cnn_pred.append(np.bincount(_votes, minlength=3).argmax())
        _offset += _n
    y_cnn_pred = np.array(y_cnn_pred)

    _cnn_acc = accuracy_score(y_test, y_cnn_pred)
    _cnn_f1  = f1_score(y_test, y_cnn_pred, average='weighted')
    results['1D-CNN'] = {
        'accuracy':         _cnn_acc,
        'f1_score':         _cnn_f1,
        'confusion_matrix': confusion_matrix(y_test, y_cnn_pred),
        'y_pred':           y_cnn_pred,
        'report':           classification_report(y_test, y_cnn_pred, output_dict=True),
    }
    print(f"\n1D-CNN (majority vote, {len(_segs_per_sig)} signals): "
          f"Accuracy={_cnn_acc:.4f}, F1={_cnn_f1:.4f}")

## 6. Final Train / Test Evaluation

Summarise accuracy and F1-score for every model on the held-out test set,
sorted by accuracy descending. The best model is identified for downstream use.

In [ ]:
# =========================================================
# 6. Final Train / Test Evaluation
# =========================================================

print(f"{'Model':<20} {'Accuracy':>10} {'F1 (macro)':>12}")
print("-" * 44)
for name, r in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"{name:<20} {r['accuracy']:>10.4f} {r['f1_score']:>12.4f}")

best_model_name = max(results, key=lambda n: results[n]['accuracy'])
print(f"\nBest model: {best_model_name}  "
      f"(accuracy={results[best_model_name]['accuracy']:.4f})")

### 6b. Binary Evaluation — Healthy vs Fault

In a real deployment the primary safety question is often binary:
**is this bearing healthy or does it need attention?**

OR_damage and IR_damage are collapsed into a single `Fault` class (positive class).
Precision, Recall, and F1 are reported for the `Fault` class — the clinically
relevant metrics when a false negative (missed fault) is more costly than a false alarm.

In [ ]:
# --- 6b. Binary Evaluation — Healthy vs Fault ---
# Collapse the 3-class problem into binary: 0 = Healthy, 1 = Fault (OR or IR).
# Positive class = Fault. Reports precision, recall, F1 per model.

from sklearn.metrics import precision_score, recall_score, f1_score

# Remap: class 0 stays 0 (Healthy); classes 1 and 2 become 1 (Fault)
y_test_bin = (y_test > 0).astype(int)

rows = []
for name, pipe in pipeline_runner.fitted_pipelines.items():
    y_pred_3 = pipe.predict(X_test_fs)
    y_pred_bin = (y_pred_3 > 0).astype(int)
    rows.append({
        "Model":     name,
        "Precision": precision_score(y_test_bin, y_pred_bin, zero_division=0),
        "Recall":    recall_score(y_test_bin,    y_pred_bin, zero_division=0),
        "F1":        f1_score(y_test_bin,        y_pred_bin, zero_division=0),
    })

binary_results = (
    pd.DataFrame(rows)
      .set_index("Model")
      .sort_values("F1", ascending=False)
      .round(4)
)
display(binary_results)

## 7. Visualisations

Confusion matrices for all classifiers on the held-out test set (n = 140 signals).
The grid includes the tuned GBT, the Stacking meta-ensemble, and the 1D-CNN
(if PyTorch was available). A perfect classifier produces a diagonal matrix;
off-diagonal entries reveal which fault types are confused.

In [ ]:
# =========================================================
# 7. Visualisations
# =========================================================

# --- 7a. Confusion Matrix Grid (all models) ---
_model_order = [n for n in ['RF', 'GBT', 'XGB', 'Stacking', '1D-CNN']
                if n in results]

_n_cols = 2
_n_rows = (len(_model_order) + _n_cols - 1) // _n_cols

fig, axes = plt.subplots(_n_rows, _n_cols, figsize=(14, 5 * _n_rows))
axes = axes.flatten()

for ax, name in zip(axes, _model_order):
    cm_mat = results[name]['confusion_matrix']
    acc    = results[name]['accuracy']
    sns.heatmap(
        cm_mat,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
        cbar=False,
        linewidths=0.5,
    )
    ax.set_title(f"{name}  (acc={acc:.3f})", fontsize=9)
    ax.set_xlabel("Predicted", fontsize=8)
    ax.set_ylabel("True", fontsize=8)
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.tick_params(axis='y', labelsize=7, rotation=0)

# Hide unused subplot slots
for ax in axes[len(_model_order):]:
    ax.set_visible(False)

fig.suptitle("Confusion Matrices — All Models (Test Set)", fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 7b. MLflow Experiment Logging ---
# Logs parameters, metrics, and the best fitted pipeline to MLflow.
# Plots are kept in PLOTS_DIR (repo-friendly); they are NOT duplicated into mlruns/.
# Run `mlflow ui --backend-store-uri mlruns` in the project root to browse results.
import pickle as _pickle

# Always persist per-condition signal normalisation stats to mlruns/ so Docker
# can apply the same signal normalisation during inference.
_sig_stats_path = _BASE_DIR / "mlruns" / "cond_signal_stats.pkl"
_sig_stats_path.parent.mkdir(parents=True, exist_ok=True)
with open(_sig_stats_path, "wb") as _f:
    _pickle.dump(_cond_signal_stats, _f)
print(f"Per-condition signal stats saved → {_sig_stats_path.name}")

if not RUN_MLFLOW:
    print("MLflow logging skipped (RUN_MLFLOW=False).")
else:
    _mlruns_path = str(_BASE_DIR / "mlruns")
    mlflow.set_tracking_uri(f"file:///{_mlruns_path}")
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    _run_name = f"{SETTING_FILTER or 'ALL'}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    with mlflow.start_run(run_name=_run_name):

        # Dataset & pipeline parameters
        mlflow.log_params({
            "bearing_set":         _set_name,
            "setting_filter":      SETTING_FILTER or "ALL",
            "n_train_signals":     len(train_idx),
            "n_test_signals":      len(test_idx),
            "n_features_raw":      X_train.shape[1],
            "n_features_selected": X_train_fs.shape[1],
            "random_state":        RANDOM_STATE,
            "test_size":           TEST_SIZE,
            "n_cv_splits":         N_SPLITS,
        })

        # Tuned hyperparameters for each model (logged only when tuning cells were run)
        _tuning_map = {
            "gbt": ("_best",      "gbt__"),
            "rf":  ("_best_rf",   "rf__"),
            "xgb": ("_best_xgb",  "xgb__"),
        }
        for _model_tag, (_var_name, _prefix) in _tuning_map.items():
            if _var_name in vars():
                _params = vars()[_var_name]
                mlflow.log_params(
                    {f"{_model_tag}_{k.replace(_prefix, '')}": v
                     for k, v in _params.items()}
                )

        # Per-model metrics
        for _name, _res in results.items():
            _prefix = _name.lower().replace("-", "_")
            mlflow.log_metric(f"{_prefix}_accuracy",    _res["accuracy"])
            mlflow.log_metric(f"{_prefix}_f1_weighted", _res["f1_score"])

        # Best model summary
        _best_model_name = max(results, key=lambda k: results[k]["accuracy"])
        mlflow.log_param( "best_model",       _best_model_name)
        mlflow.log_metric("best_accuracy",    results[_best_model_name]["accuracy"])
        mlflow.log_metric("best_f1_weighted", results[_best_model_name]["f1_score"])

        # Save inference pipeline: already a full Pipeline(scaler + model) after refactor.
        # Inference flow: raw signal -> signal_norm -> feature_extract -> freq_norm -> selector -> pipeline.predict()
        if hasattr(pipeline_runner, "fitted_pipelines") and \
                _best_model_name in pipeline_runner.fitted_pipelines:
            _inference_pipeline = pipeline_runner.fitted_pipelines[_best_model_name]
            mlflow.sklearn.log_model(
                _inference_pipeline,
                artifact_path="best_model",
                registered_model_name=f"bearing_fault_{_best_model_name.lower()}",
            )
            # Register selector so inference_api.py can load it from the Registry
            mlflow.sklearn.log_model(
                selector,
                artifact_path="feature_selector",
                registered_model_name="bearing_fault_selector",
            )
            print(f"  Model saved  : '{_best_model_name}'  (scaler + classifier)")
            print(f"  Selector     : registered as 'bearing_fault_selector'")

        print(f"MLflow run logged: {MLFLOW_EXPERIMENT} / {_run_name}")
        print(f"  Tracking URI : {_mlruns_path}")
        print(f"  Best model   : {_best_model_name}  "
              f"acc={results[_best_model_name]['accuracy']:.4f}  "
              f"F1={results[_best_model_name]['f1_score']:.4f}")
        print("  View UI      : mlflow ui")

## 8. Explainability

SHAP `TreeExplainer` is used to attribute each prediction to individual features.
`TreeExplainer` works directly with tree-based models (RF, XGB) without
approximation — it computes exact Shapley values in polynomial time.

`shap_vals` has shape `(n_samples, n_features, n_classes)` for multi-class tree models.
The beeswarm summary plot shows the top features by mean |SHAP| across all classes,
revealing which DSP features drive fault discrimination the most.

SHAP is computed on `X_test` only — no training data is touched (no leakage).

In [ ]:
# =========================================================
# 8. Explainability
# =========================================================

# GradientBoostingClassifier (GBT) is not supported by SHAP TreeExplainer for
# multi-class problems — exclude it and fall back to RF or XGB.
_SHAP_UNSUPPORTED = {'GBT', 'Stacking'}
_TREE_PRIORITY    = ['RF', 'XGB']

shap_model_name = next(
    (m for m in [best_model_name] + _TREE_PRIORITY
     if m in pipeline_runner.pipelines
     and m not in _SHAP_UNSUPPORTED
     and hasattr(pipeline_runner.pipelines[m].named_steps['model'], 'feature_importances_')),
    None,
)

if shap_model_name is not None:
    # Extract the fitted classifier and its scaler from the pipeline
    fitted_model  = pipeline_runner.pipelines[shap_model_name].named_steps['model']
    fitted_scaler = pipeline_runner.pipelines[shap_model_name].named_steps['scaler']

    # Use held-out test data only — never training data (no leakage).
    # X_test_fs is feature-selected; the pipeline scaler applies the final
    # global z-score so that the feature space matches training exactly.
    n_shap = min(SHAP_SAMPLE_SIZE, len(X_test_fs))
    X_shap        = X_test_fs[:n_shap]
    X_shap_scaled = fitted_scaler.transform(X_shap)

    explainer = shap.TreeExplainer(fitted_model)
    shap_vals = explainer.shap_values(X_shap_scaled)   # shape depends on SHAP version

    # Newer SHAP (>=0.40) returns a 3D array (n_samples, n_features, n_classes);
    # older SHAP returns a list of (n_samples, n_features) arrays — handle both.
    if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        shap_vals_list = [shap_vals[:, :, i] for i in range(shap_vals.shape[2])]
    else:
        shap_vals_list = shap_vals   # already a list

    # --- 8a. SHAP Summary Plot (all classes, top 20 features) ---
    _palette = np.linspace(0.30, 0.90, len(class_names))
    _shap_color_fn = lambda i: CMAP(_palette[i])

    plt.figure(figsize=FigSize.HEATMAP_LARGE)
    shap.summary_plot(
        shap_vals_list, X_shap_scaled,
        feature_names=feature_names_fs,
        class_names=class_names,
        color=_shap_color_fn,
        show=False,
    )
    plt.title(f"SHAP Feature Importance — {shap_model_name} (n={n_shap})")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "06_shap_summary.png", dpi=150, bbox_inches="tight")
    plt.show()

    # --- 8b. Per-class mean |SHAP| table (top 10 features) ---
    shap_df_list = []
    for cls_idx, cls_name in enumerate(class_names):
        vals = np.abs(shap_vals_list[cls_idx])      # (n_shap, n_features)
        mean_abs = vals.mean(axis=0)                # (n_features,)
        shap_df_list.append(
            pd.Series(mean_abs, index=feature_names_fs, name=cls_name)
        )
    shap_summary = pd.DataFrame(shap_df_list).T
    shap_summary['mean_all_classes'] = shap_summary.mean(axis=1)
    shap_summary = shap_summary.sort_values('mean_all_classes', ascending=False)

    print(f"\nTop 10 features by mean |SHAP| ({shap_model_name}):")
    display(shap_summary.head(10).style.format("{:.4f}"))
else:
    print("No compatible tree-based model available for SHAP TreeExplainer; skipping.")

## 9. Summary & Conclusions

**Task:** 3-class bearing fault classification (Healthy / Outer Race Damage / Inner Race
Damage) on the Paderborn University dataset (32 bearings, all four operating conditions)
using 183 DSP features extracted from phase-current and vibration signals.

**Best model:** determined at runtime — see Section 6 output.

**Key drivers** (from SHAP — Section 8):
- Envelope features at BPFO / BPFI frequencies capture fault-impulse repetition rates
- RMS and kurtosis of phase current distinguish healthy from damaged bearings
- Wavelet packet sub-band energies differentiate outer-race from inner-race patterns

**Full pipeline:**

| Step | Details |
|---|---|
| Data acquisition | 32 bearings × 4 operating conditions via `ensure_data` |
| Signal normalisation | Per-condition z-score on raw vibration / current (training stats only) |
| DSP feature extraction | 183 features — time, frequency, WPD, envelope (cached) |
| Frequency normalisation | Hz → shaft orders for spectral features |
| Feature selection | `SelectFromModel` (RF, median threshold): 183 → ~92 features |
| CV strategy | `StratifiedGroupKFold` — bearing-aware, prevents memorisation of individual bearings |
| Hyperparameter tuning | `RandomizedSearchCV` (30 iters, 3-fold inner CV) for GBT, RF, XGB |
| ML classification | 3 models + Stacking meta-ensemble (RF + GBT + XGB → LR) |
| Leakage prevention | sklearn `Pipeline` wraps scaler + model; re-fit inside every CV fold |
| 1D-CNN | Architecture defined; requires GPU for practical training times |
| Explainability | SHAP `TreeExplainer` on best tree-based model |
| Experiment tracking | MLflow — params, metrics, best model saved to registry |
| Deployment | FastAPI + Docker inference service (`/predict_mat` endpoint) |

**Limitations & next steps:**
1. IR vs OR confusion remains the primary error — 1x/2x/3x BPFI/BPFO harmonics exist;
   adding 4x/5x harmonics or ICS2 cyclostationarity indicators may improve separation.
2. Evaluate cross-condition generalisation — train on one operating condition, test on another.
3. Explore artificial → real damage domain adaptation (`MINIMAL_SET` subset).